Finetune the minilm model to be able to improve the quality of sentence embeddings we get with SentenceTransformers (SBERT)

In [1]:
 # Load dataset
from local_utilities.dataset import get_dataset_train

human_texts, ai_texts, _ = get_dataset_train(randomize=False)

[nltk_data] Downloading package punkt_tab to /home/tobias/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/home/tobias/.pyenv/versions/genai/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
 # Label training data
from sentence_transformers import InputExample
train_data = []

# Label human texts
for text in human_texts:
    train_data.append(InputExample(texts=[text], label=0.0))

# Label AI texts
for text in ai_texts:
    train_data.append(InputExample(texts=[text], label=1.0))

In [3]:
 # Load model
from local_utilities.model import get_minilm_model
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(get_minilm_model())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1894.09it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
 # Set up dataloader
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    train_data,
    shuffle=True,
    batch_size=320
)

In [5]:
 # Set up loss function
from sentence_transformers import losses
train_loss = losses.BatchHardSoftMarginTripletLoss(model = model)

In [6]:
# Train model
model.fit(
    train_objectives = [(train_dataloader, train_loss)],
    epochs = 3,
    warmup_steps = 600,
    use_amp = True,
    show_progress_bar = True
)

Step,Training Loss
500,0.815903
1000,0.713036
1500,0.707980
2000,0.706282
2500,0.704896
3000,0.703915
3500,0.702945
4000,0.702213
4500,0.701654
5000,0.701124


In [ ]:
# Store retrained model
from local_utilities.directory import get_minilm_directory
model.save(get_minilm_directory())

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.87it/s]


: 